In [1]:
import pandas as pd
from pathlib import Path


AUTHENTIC_ROOT = Path("dataset_split/train/authentic")
FORENSIC_ROOT = Path("dataset_split/train/forged")

material_classes = {
    "Biological",
    "Fibres",
    "Nanowires",
    "Particles",
    "Porous_Sponge",
    "Powder"
}

device_classes = {
    "Films_Coated_Surface",
    "MEMS_devices_and_electrodes",
    "Patterned_surface",
    "Tips"
}


records = []

# -----------------------------------
# Add Authentic Images
# -----------------------------------

for class_dir in AUTHENTIC_ROOT.iterdir():

    if not class_dir.is_dir():
        continue

    defect_type = class_dir.name

    if defect_type in material_classes:
        domain = "Material"
    else:
        domain = "Device"

    for img_path in class_dir.glob("*"):

        records.append({

            "image_name":
                img_path.name,

            "image_path":
                str(img_path),

            "defect_type":
                defect_type,

            "domain":
                domain,

            "authenticity":
                "Authentic",

            "mask_path":
                ""

        })


# -----------------------------------
# Add Forged Images
# -----------------------------------

for forgery_type_dir in FORENSIC_ROOT.iterdir():

    if not forgery_type_dir.is_dir():
        continue

    forgery_type = forgery_type_dir.name

    for class_dir in forgery_type_dir.iterdir():

        if not class_dir.is_dir():
            continue

        defect_type = class_dir.name

        domain = (
            "Material"
            if defect_type in material_classes
            else "Device"
        )

        for img_path in class_dir.glob("*"):

            mask_path = str(

                Path(
                    "dataset_split/train/masks"
                )

                / forgery_type
                / defect_type
                / img_path.name
            )

            records.append({

                "image_name":
                    img_path.name,

                "image_path":
                    str(img_path),

                "defect_type":
                    defect_type,

                "domain":
                    domain,

                "authenticity":
                    forgery_type,

                "mask_path":
                    mask_path
            })
# -----------------------------------
# Create DataFrame
# -----------------------------------

df = pd.DataFrame(records)

print(df.head())

# -----------------------------------
# Save CSV
# -----------------------------------

df.to_csv(
    "SEM_forensic_metadata.csv",
    index=False
)

print(
    f"\nTotal records: {len(df)}"
)

print(
    "\nSaved as SEM_forensic_metadata.csv"
)

                                image_name  \
0  L4_c0881a70509cbc98a7e8eff93d86f2f9.jpg   
1  L4_29ad5d70767fc5a51b0ce4f232fae40e.jpg   
2  L4_80ed47ecd9b707cf5210fd548f7c7286.jpg   
3  L4_a32d5b70c1374299f1866f966f3c081a.jpg   
4  L4_b3e693c0c3ab2d746d6600bcd8981c44.jpg   

                                          image_path defect_type    domain  \
0  dataset_split/train/authentic/Powder/L4_c0881a...      Powder  Material   
1  dataset_split/train/authentic/Powder/L4_29ad5d...      Powder  Material   
2  dataset_split/train/authentic/Powder/L4_80ed47...      Powder  Material   
3  dataset_split/train/authentic/Powder/L4_a32d5b...      Powder  Material   
4  dataset_split/train/authentic/Powder/L4_b3e693...      Powder  Material   

  authenticity mask_path  
0    Authentic            
1    Authentic            
2    Authentic            
3    Authentic            
4    Authentic            

Total records: 3600

Saved as SEM_forensic_metadata.csv


In [2]:
#pip install torch torchvision timm opencv-python scikit-image pillow

In [3]:
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from PIL import Image
from skimage.feature import local_binary_pattern

import timm

from torchvision.models import efficientnet_b0

import pandas as pd

import torch
from torch.utils.data import Dataset
from torchvision import transforms
from torch.utils.data import DataLoader

In [4]:
from io import BytesIO
class ForensicPreprocessor:

    def __init__(self,
                 lbp_radius=2,
                 lbp_points=16):

        self.radius = lbp_radius
        self.points = lbp_points

    def normalize(self, img):

        img = img.astype(np.float32)
    
        min_val = img.min()
        max_val = img.max()
    
        if max_val - min_val < 1e-8:
            return np.zeros_like(img, dtype=np.float32)
    
        return (
            img - min_val
        ) / (
            max_val - min_val
        )

    def residual_map(self, img):

        blur = cv2.GaussianBlur(
            img,
            (5,5),
            0
        )

        residual = cv2.absdiff(
            img,
            blur
        )

        return self.normalize(residual)

    def fft_map(self, img):

        f = np.fft.fft2(img)

        fshift = np.fft.fftshift(f)

        magnitude = np.log(
            np.abs(fshift)+1
        )

        return self.normalize(magnitude)

    def lbp_map(self, img):

        lbp = local_binary_pattern(
            img,
            P=self.points,
            R=self.radius,
            method='uniform'
        )

        return self.normalize(lbp)

    def ela_map(self, img):
    
        buffer = BytesIO()
    
        Image.fromarray(img).save(
            buffer,
            format="JPEG",
            quality=90
        )
    
        buffer.seek(0)
    
        recompressed = np.array(
            Image.open(buffer).convert("L")
        )
    
        ela = cv2.absdiff(
            img,
            recompressed
        )
    
        return self.normalize(ela)

    def process(self, img):

        residual = self.residual_map(img)

        fft = self.fft_map(img)

        lbp = self.lbp_map(img)

        ela = self.ela_map(img)

        stacked = np.stack([
            residual,
            fft,
            lbp,
            ela
        ], axis=0)

        return stacked

In [5]:
class CNNBranch(nn.Module):

    def __init__(self):

        super().__init__()

        backbone = efficientnet_b0(
            weights="DEFAULT"
        )

        backbone.classifier = nn.Identity()

        self.backbone = backbone

    def forward(self,x):

        return self.backbone(x)

In [6]:
import torch.nn.functional as F

class ViTBranch(nn.Module):

    def __init__(self):
        super().__init__()

        self.vit = timm.create_model(
            'vit_base_patch16_224',
            pretrained=True,
            num_classes=0
        )

    def forward(self, x):

        x = F.interpolate(
            x,
            size=(224, 224),
            mode="bilinear",
            align_corners=False
        )

        return self.vit(x)

In [7]:
class MorphologyBranch(nn.Module):

    def __init__(self):

        super().__init__()

        self.cnn = CNNBranch()

        self.vit = ViTBranch()

        self.fc = nn.Sequential(

            nn.Linear(
                1280+768,
                1024
            ),

            nn.ReLU(),

            nn.Dropout(0.3)
        )

    def forward(self,x):

        cnn_feat = self.cnn(x)

        vit_feat = self.vit(x)

        combined = torch.cat(
            [
                cnn_feat,
                vit_feat
            ],
            dim=1
        )

        morphology_feat = self.fc(
            combined
        )

        return morphology_feat

In [8]:
class ForensicCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                4,
                32,
                3,
                padding=1
            ),

            nn.BatchNorm2d(32),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                32,
                64,
                3,
                padding=1
            ),

            nn.BatchNorm2d(64),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                64,
                128,
                3,
                padding=1
            ),

            nn.BatchNorm2d(128),

            nn.ReLU(),

            nn.AdaptiveAvgPool2d(1)
        )

        self.fc = nn.Linear(
            128,
            256
        )

    def forward(self,x):

        x = self.features(x)

        x = x.view(
            x.size(0),
            -1
        )

        x = self.fc(x)

        return x

In [9]:
class FeatureFusion(nn.Module):

    def __init__(self):
        super().__init__()

        self.morph_proj = nn.Sequential(
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.ReLU()
        )

        self.forensic_proj = nn.Sequential(
            nn.Linear(256, 512),
            nn.LayerNorm(512),
            nn.ReLU()
        )

        self.gate = nn.Sequential(
            nn.Linear(1024, 512),
            nn.Sigmoid()
        )

        self.output_fc = nn.Sequential(
            nn.Linear(512, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

    def forward(
        self,
        morphology_feat,
        forensic_feat
    ):

        morphology_feat = self.morph_proj(
            morphology_feat
        )

        forensic_feat = self.forensic_proj(
            forensic_feat
        )

        combined = torch.cat(
            [
                morphology_feat,
                forensic_feat
            ],
            dim=1
        )

        gate = self.gate(combined)

        fused = (
            gate * morphology_feat
            +
            (1.0 - gate) * forensic_feat
        )

        fused = self.output_fc(fused)

        return fused

In [10]:
class SEMFeatureExtractor(nn.Module):

    def __init__(self):

        super().__init__()

        self.morphology = MorphologyBranch()

        self.forensic = ForensicCNN()

        self.fusion = FeatureFusion()

    def forward(
        self,
        sem_image,
        forensic_maps
    ):

        morphology_feat = self.morphology(
            sem_image
        )

        forensic_feat = self.forensic(
            forensic_maps
        )

        fused_feat = self.fusion(
            morphology_feat,
            forensic_feat
        )

        return {

            "morphology_features":
                morphology_feat,

            "forensic_features":
                forensic_feat,

            "fused_features":
                fused_feat
        }

In [11]:
DEFECT_MAP = {
    "Biological":0,
    "Fibres":1,
    "Films_Coated_Surface":2,
    "MEMS_devices_and_electrodes":3,
    "Nanowires":4,
    "Particles":5,
    "Patterned_surface":6,
    "Porous_Sponge":7,
    "Powder":8,
    "Tips":9
}

DOMAIN_MAP = {
    "Material":0,
    "Device":1
}

AUTH_MAP = {
    "Authentic":0,
    "copy_move":1,
    "splicing":2,
    "synthetic":3,
    "contrast":4
}

In [12]:
class SEMForensicDataset(Dataset):

    def __init__(
        self,
        csv_file,
        forensic_preprocessor
        # forensic_preprocessor,
        # image_size=224
    ):

        self.df = pd.read_csv(csv_file)

        self.preprocessor = forensic_preprocessor

        self.transform = transforms.Compose([

            transforms.ToPILImage(),

            # transforms.Resize(
            #     (image_size,image_size)
            # ),

            transforms.Grayscale(
                num_output_channels=3
            ),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.5, 0.5, 0.5],
                std=[0.5, 0.5, 0.5]
            )
        ])

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img_path = row["image_path"]

        img = cv2.imread(
            img_path,
            cv2.IMREAD_GRAYSCALE
        )

        # img_resized = cv2.resize(
        #     img,
        #     (224,224)
        # )

        sem_tensor = self.transform(
            #img_resized
            img
        )

        forensic_maps = self.preprocessor.process(
            #img_resized
            img
        )

        forensic_tensor = torch.tensor(
            forensic_maps,
            dtype=torch.float32
        )

        defect_label = DEFECT_MAP[
            row["defect_type"]
        ]

        domain_label = DOMAIN_MAP[
            row["domain"]
        ]

        auth_label = AUTH_MAP[
            row["authenticity"]
        ]

        sample = {

            "sem_image":
                sem_tensor,

            "forensic_maps":
                forensic_tensor,

            "defect":
                torch.tensor(
                    defect_label
                ),

            "domain":
                torch.tensor(
                    domain_label
                ),

            "auth":
                torch.tensor(
                    auth_label
                )
        }

        return sample

In [13]:
preprocessor = ForensicPreprocessor()

dataset = SEMForensicDataset(

    csv_file=
    "SEM_forensic_metadata.csv",

    forensic_preprocessor=
    preprocessor
)

In [14]:
train_loader = DataLoader(

    dataset,

    batch_size=16,

    shuffle=True,

    num_workers=0
)

In [15]:
class MultiTaskClassifier(nn.Module):

    def __init__(self,
                 input_dim=512):

        super().__init__()

        # Shared representation

        self.shared = nn.Sequential(

            nn.Linear(input_dim, 256),

            nn.BatchNorm1d(256),

            nn.ReLU(),

            nn.Dropout(0.3)
        )

        # -----------------------
        # Defect Head
        # -----------------------

        self.defect_head = nn.Sequential(

            nn.Linear(256,128),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(128,10)
        )

        # -----------------------
        # Domain Head
        # -----------------------

        self.domain_head = nn.Sequential(

            nn.Linear(256,64),

            nn.ReLU(),

            nn.Linear(64,2)
        )

        # -----------------------
        # Authenticity Head
        # -----------------------

        self.auth_head = nn.Sequential(

            nn.Linear(256,128),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(128,5)
        )

    def forward(self,x):

        shared_feat = self.shared(x)

        defect_logits = self.defect_head(
            shared_feat
        )

        domain_logits = self.domain_head(
            shared_feat
        )

        auth_logits = self.auth_head(
            shared_feat
        )

        return {

            "defect_logits":
                defect_logits,

            "domain_logits":
                domain_logits,

            "auth_logits":
                auth_logits
        }

In [16]:
class ForensicSEMModel(nn.Module):

    def __init__(self,
                 feature_extractor):

        super().__init__()

        self.feature_extractor = feature_extractor

        self.classifier = MultiTaskClassifier()

    def forward(
        self,
        sem_image,
        forensic_maps
    ):

        features = self.feature_extractor(
            sem_image,
            forensic_maps
        )
    
        fused_features = features[
            "fused_features"
        ]
    
        outputs = self.classifier(
            fused_features
        )
    
        return outputs

In [17]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

feature_extractor = SEMFeatureExtractor()

model = ForensicSEMModel(
    feature_extractor
).to(device)

In [18]:
from sklearn.utils.class_weight import compute_class_weight

train_defect_labels = []
train_domain_labels = []
train_auth_labels = []

for batch in train_loader:

    train_defect_labels.extend(
        batch["defect"].cpu().numpy()
    )

    train_domain_labels.extend(
        batch["domain"].cpu().numpy()
    )

    train_auth_labels.extend(
        batch["auth"].cpu().numpy()
    )

train_defect_labels = np.array(train_defect_labels)
train_domain_labels = np.array(train_domain_labels)
train_auth_labels = np.array(train_auth_labels)

defect_classes = np.unique(train_defect_labels)

defect_weights = compute_class_weight(
    class_weight="balanced",
    classes=defect_classes,
    y=train_defect_labels
)

defect_weights = torch.tensor(
    defect_weights,
    dtype=torch.float32
).to(device)

domain_classes = np.unique(train_domain_labels)

domain_weights = compute_class_weight(
    class_weight="balanced",
    classes=domain_classes,
    y=train_domain_labels
)

domain_weights = torch.tensor(
    domain_weights,
    dtype=torch.float32
).to(device)

auth_classes = np.unique(train_auth_labels)

auth_weights = compute_class_weight(
    class_weight="balanced",
    classes=auth_classes,
    y=train_auth_labels
)

auth_weights = torch.tensor(
    auth_weights,
    dtype=torch.float32
).to(device)

In [19]:
from sklearn.utils.class_weight import compute_class_weight

def compute_loss(
    outputs,
    defect_labels,
    domain_labels,
    auth_labels
):
       
    #defect_loss_fn = nn.CrossEntropyLoss(weight=defect_weights)
    defect_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)

    loss_defect = defect_loss_fn(
        outputs["defect_logits"],
        defect_labels
    )
       
    #domain_loss_fn = nn.CrossEntropyLoss(weight=domain_weights)
    domain_loss_fn = nn.CrossEntropyLoss()
 
    loss_domain = domain_loss_fn(
        outputs["domain_logits"],
        domain_labels
    )

    auth_loss_fn = nn.CrossEntropyLoss(weight=auth_weights)

    loss_auth = auth_loss_fn(
        outputs["auth_logits"],
        auth_labels
    )

    total_loss = (

          loss_defect

        + loss_domain

        + loss_auth
    )

    return total_loss

In [20]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50,
    eta_min=1e-6
)

In [21]:
num_epochs = 50

for epoch in range(num_epochs):

    model.train()
    running_loss = 0.0

    for batch in train_loader:

        sem_images = batch["sem_image"].to(device)
        forensic_maps = batch["forensic_maps"].to(device)

        defect_labels = batch["defect"].long().to(device)
        domain_labels = batch["domain"].long().to(device)
        auth_labels = batch["auth"].long().to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            sem_images,
            forensic_maps
        )

        loss = compute_loss(
            outputs,
            defect_labels,
            domain_labels,
            auth_labels
        )

        loss.backward()

        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

       
        optimizer.step()

        running_loss += loss.item()

    
    scheduler.step()

    average_loss = running_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {average_loss:.4f} "
        f"LR: {optimizer.param_groups[0]['lr']:.7f}"
    )

Epoch [1/50] Loss: 3.9629 LR: 0.0002997
Epoch [2/50] Loss: 2.9052 LR: 0.0002988
Epoch [3/50] Loss: 2.5573 LR: 0.0002974
Epoch [4/50] Loss: 2.2742 LR: 0.0002953
Epoch [5/50] Loss: 2.1305 LR: 0.0002927
Epoch [6/50] Loss: 1.7224 LR: 0.0002895
Epoch [7/50] Loss: 1.4147 LR: 0.0002858
Epoch [8/50] Loss: 1.2896 LR: 0.0002815
Epoch [9/50] Loss: 1.1565 LR: 0.0002767
Epoch [10/50] Loss: 1.1168 LR: 0.0002714
Epoch [11/50] Loss: 1.0462 LR: 0.0002657
Epoch [12/50] Loss: 0.9811 LR: 0.0002595
Epoch [13/50] Loss: 0.9360 LR: 0.0002528
Epoch [14/50] Loss: 0.9247 LR: 0.0002458
Epoch [15/50] Loss: 0.8879 LR: 0.0002384
Epoch [16/50] Loss: 0.8364 LR: 0.0002306
Epoch [17/50] Loss: 0.8026 LR: 0.0002225
Epoch [18/50] Loss: 0.7689 LR: 0.0002142
Epoch [19/50] Loss: 0.7307 LR: 0.0002055
Epoch [20/50] Loss: 0.6964 LR: 0.0001967
Epoch [21/50] Loss: 0.6829 LR: 0.0001877
Epoch [22/50] Loss: 0.6477 LR: 0.0001785
Epoch [23/50] Loss: 0.6215 LR: 0.0001692
Epoch [24/50] Loss: 0.6062 LR: 0.0001599
Epoch [25/50] Loss: 0.586

In [22]:
AUTHENTIC_ROOT = Path("dataset_split/test/authentic")
FORENSIC_ROOT = Path("dataset_split/test/forged")

material_classes = {
    "Biological",
    "Fibres",
    "Nanowires",
    "Particles",
    "Porous_Sponge",
    "Powder"
}

device_classes = {
    "Films_Coated_Surface",
    "MEMS_devices_and_electrodes",
    "Patterned_surface",
    "Tips"
}


records = []

# -----------------------------------
# Add Authentic Images
# -----------------------------------

for class_dir in AUTHENTIC_ROOT.iterdir():

    if not class_dir.is_dir():
        continue

    defect_type = class_dir.name

    if defect_type in material_classes:
        domain = "Material"
    else:
        domain = "Device"

    for img_path in class_dir.glob("*"):

        records.append({

            "image_name":
                img_path.name,

            "image_path":
                str(img_path),

            "defect_type":
                defect_type,

            "domain":
                domain,

            "authenticity":
                "Authentic",

            "mask_path":
                ""

        })


# -----------------------------------
# Add Forged Images
# -----------------------------------

for forgery_type_dir in FORENSIC_ROOT.iterdir():

    if not forgery_type_dir.is_dir():
        continue

    forgery_type = forgery_type_dir.name

    for class_dir in forgery_type_dir.iterdir():

        if not class_dir.is_dir():
            continue

        defect_type = class_dir.name

        domain = (
            "Material"
            if defect_type in material_classes
            else "Device"
        )

        for img_path in class_dir.glob("*"):

            mask_path = str(

                Path(
                    "dataset_split/test/masks"
                )

                / forgery_type
                / defect_type
                / img_path.name
            )

            records.append({

                "image_name":
                    img_path.name,

                "image_path":
                    str(img_path),

                "defect_type":
                    defect_type,

                "domain":
                    domain,

                "authenticity":
                    forgery_type,

                "mask_path":
                    mask_path
            })
# -----------------------------------
# Create DataFrame
# -----------------------------------

df = pd.DataFrame(records)

print(df.head())

# -----------------------------------
# Save CSV
# -----------------------------------

df.to_csv(
    "SEM_forensic_metadata_test.csv",
    index=False
)

print(
    f"\nTotal records: {len(df)}"
)

print(
    "\nSaved as SEM_forensic_metadata.csv"
)

                                image_name  \
0  L4_31f600bbeaedd8e5864d0bdb7ae90b1a.jpg   
1  L4_87b7aae0859a54360b0ba7d96cf1a6a0.jpg   
2  L4_abcdfcc55a588be31b4d95b317820250.jpg   
3  L4_3e2bcd1e9d82cca5b97116aa46adbde2.jpg   
4  L4_1bba0cf4a9f6e655e71b4430ca0ee22d.jpg   

                                          image_path defect_type    domain  \
0  dataset_split/test/authentic/Powder/L4_31f600b...      Powder  Material   
1  dataset_split/test/authentic/Powder/L4_87b7aae...      Powder  Material   
2  dataset_split/test/authentic/Powder/L4_abcdfcc...      Powder  Material   
3  dataset_split/test/authentic/Powder/L4_3e2bcd1...      Powder  Material   
4  dataset_split/test/authentic/Powder/L4_1bba0cf...      Powder  Material   

  authenticity mask_path  
0    Authentic            
1    Authentic            
2    Authentic            
3    Authentic            
4    Authentic            

Total records: 900

Saved as SEM_forensic_metadata.csv


In [23]:
preprocessor = ForensicPreprocessor()

test_dataset = SEMForensicDataset(

    csv_file=
    "SEM_forensic_metadata_test.csv",

    forensic_preprocessor=
    preprocessor
)

In [24]:
test_loader = DataLoader(

    test_dataset,

    batch_size=16,

    shuffle=False,

    num_workers=0
)

In [25]:
import torch
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

In [26]:
def evaluate_model(
        model,
        test_loader,
        device):

    model.eval()

    defect_true = []
    defect_pred = []

    domain_true = []
    domain_pred = []

    auth_true = []
    auth_pred = []

    with torch.no_grad():

        for batch in test_loader:

            sem_images = batch[
                "sem_image"
            ].to(device)

            forensic_maps = batch[
                "forensic_maps"
            ].to(device)

            outputs = model(
                sem_images,
                forensic_maps
            )

            defect_out = torch.argmax(
                outputs["defect_logits"],
                dim=1
            )

            domain_out = torch.argmax(
                outputs["domain_logits"],
                dim=1
            )

            auth_out = torch.argmax(
                outputs["auth_logits"],
                dim=1
            )

            defect_true.extend(
                batch["defect"].numpy()
            )

            defect_pred.extend(
                defect_out.cpu().numpy()
            )

            domain_true.extend(
                batch["domain"].numpy()
            )

            domain_pred.extend(
                domain_out.cpu().numpy()
            )

            auth_true.extend(
                batch["auth"].numpy()
            )

            auth_pred.extend(
                auth_out.cpu().numpy()
            )

    results = {}

    # -----------------------
    # Defect Classification
    # -----------------------

    acc = accuracy_score(
        defect_true,
        defect_pred
    )

    prec, rec, f1, _ = \
        precision_recall_fscore_support(
            defect_true,
            defect_pred,
            average='weighted',
            zero_division=0
        )

    results["defect"] = {

        "accuracy": acc,

        "precision": prec,

        "recall": rec,

        "f1": f1,

        "cm":
            confusion_matrix(
                defect_true,
                defect_pred
            )
    }

    # -----------------------
    # Domain Classification
    # -----------------------

    acc = accuracy_score(
        domain_true,
        domain_pred
    )

    prec, rec, f1, _ = \
        precision_recall_fscore_support(
            domain_true,
            domain_pred,
            average='weighted',
            zero_division=0
        )

    results["domain"] = {

        "accuracy": acc,

        "precision": prec,

        "recall": rec,

        "f1": f1,

        "cm":
            confusion_matrix(
                domain_true,
                domain_pred
            )
    }

    # -----------------------
    # Authenticity
    # -----------------------

    acc = accuracy_score(
        auth_true,
        auth_pred
    )

    prec, rec, f1, _ = \
        precision_recall_fscore_support(
            auth_true,
            auth_pred,
            average='weighted',
            zero_division=0
        )

    results["auth"] = {

        "accuracy": acc,

        "precision": prec,

        "recall": rec,

        "f1": f1,

        "cm":
            confusion_matrix(
                auth_true,
                auth_pred
            )
    }

    return results

In [27]:
results = evaluate_model(
    model,
    test_loader,
    device
)

In [28]:
for task in results:

    print("\n")
    print("="*50)
    print(task.upper())
    print("="*50)

    print(
        "Accuracy:",
        results[task]["accuracy"]
    )

    print(
        "Precision:",
        results[task]["precision"]
    )

    print(
        "Recall:",
        results[task]["recall"]
    )

    print(
        "F1:",
        results[task]["f1"]
    )

    print("\nConfusion Matrix")

    print(
        results[task]["cm"]
    )



DEFECT
Accuracy: 0.8555555555555555
Precision: 0.8616506839067442
Recall: 0.8555555555555555
F1: 0.8561667117003539

Confusion Matrix
[[82  0  0  0  6  0  0  0  2  0]
 [ 4 81  0  3  0  0  0  0  0  2]
 [ 3  0 75  2  0  2  0  1  7  0]
 [ 0  0  0 70  0  0 12  0  0  8]
 [ 0  1  7  0 67  3  2  2  0  8]
 [ 0  0  8  0  0 77  2  0  2  1]
 [ 0  3  5  1  2  5 71  1  1  1]
 [ 0  1  4  0  0  0  0 85  0  0]
 [ 0  0 11  0  0  3  0  0 76  0]
 [ 0  0  0  1  2  1  0  0  0 86]]


DOMAIN
Accuracy: 0.9155555555555556
Precision: 0.9171659919028341
Recall: 0.9155555555555556
F1: 0.9159000509943905

Confusion Matrix
[[492  48]
 [ 28 332]]


AUTH
Accuracy: 0.8988888888888888
Precision: 0.8988052838775417
Recall: 0.8988888888888888
F1: 0.8969409291635877

Confusion Matrix
[[277  20   1   0   2]
 [ 50  95   2   0   1]
 [  1   7 132   0   0]
 [  0   0   0 149   0]
 [  6   1   0   0 156]]
